In [ ]:
import numpy as np
from numpy.typing import NDArray

arr = np.array

In [ ]:
from matplotlib.cm import tab10
import ipywidgets as pw

import pyvista as pv

# pv.set_jupyter_backend("server")
tab10colors = tab10.colors  # type: ignore

## B-spline matrices

For uniform knots $t_i = i$ and an interval: $i \le t \lt i+1$, localized $t' = t - i \implies 0 \le t' \lt 1$

$$
\begin{bmatrix} B^{(d)}_{i-d} & \cdots & B^{(d)}_{i} \end{bmatrix} =
\begin{bmatrix} 1 & t' & \cdots & t'^d \end{bmatrix} M^{(d)}
$$

$$
M^{(1)} =
\begin{bmatrix}
1 & 0 \\
-1 & 1
\end{bmatrix}
$$

$$
M^{(2)} =
\frac{1}{2}
\begin{bmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{bmatrix}
$$

$$
M^{(3)} =
\frac{1}{6}
\begin{bmatrix}
1 & 4 & 1 & 0 \\
-3 & 0 & 3 & 0 \\
3 & -6 & 3 & 0 \\
-1 & 3 & -3 & 1
\end{bmatrix}
$$


All the stuff doesn't depend on $i$

$
\begin{bmatrix} B^{(1)}_{i-1} & B^{(1)}_{i} \end{bmatrix} = 
\begin{bmatrix} 1 & t' \end{bmatrix}
\begin{bmatrix}
1 & 0 \\
-1 & 1
\end{bmatrix} = 
\big[ 1 - t', t' \big]
$

$
\begin{bmatrix} B^{(2)}_{i-2} & B^{(2)}_{i-1} & B^{(2)}_{i} \end{bmatrix} =
\frac{1}{2}
\begin{bmatrix} 1 & t' & t'^2 \end{bmatrix}
\begin{bmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{bmatrix} =
\frac{1}{2} \big[1 - 2 t' + t'^2, 1 + 2 t' - 2 t'^2, t'^2 \big]
$


## Curve

With control points:

$$
C^{(d)}_i = \begin{bmatrix}
c_{i-d} \\
c_{i-d+1} \\
\vdots \\
c_{i}
\end{bmatrix}
$$

$$
\mathcal{C}^{(d)}_i(t) = \begin{bmatrix} 1 & t' & \cdots & t'^d \end{bmatrix} M^{(d)} C^{(d)}_i
$$


The $M C_i$ is constant within interval

$
M^{(1)} C^{(1)}_i =
\begin{bmatrix}
1 & 0 \\
-1 & 1
\end{bmatrix}
\begin{bmatrix}
c_{i-1} \\
c_i
\end{bmatrix} =
\begin{bmatrix}
c_{i - 1} \\
c_{i} - c_{i - 1}
\end{bmatrix}
$

$
M^{(2)} C^{(2)}_i = 
\frac{1}{2}
\begin{bmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{bmatrix}
\begin{bmatrix}
c_{i-2} \\
c_{i-1} \\
c_i
\end{bmatrix} =
\frac{1}{2}
\begin{bmatrix}
c_{i-2} + c_{i-1} \\
-2 c_{i-2} + 2 c_{i-1} \\
c_{i-2} - 2c_{i-1} + c_{i}
\end{bmatrix}
$


---


#### The $M^{(d)}$


In [ ]:
# blending matrices for uniform splines
M2 = np.array([[1, 1, 0], [-2, 2, 0], [1, -2, 1]]) / 2
M3 = np.array([[1, 4, 1, 0], [-3, 0, 3, 0], [3, -6, 3, 0], [-1, 3, -3, 1]]) / 6

#### The $P^{(d)}(t)$

$= \begin{bmatrix} 1 & t & t^2 & \cdots & t^d \end{bmatrix}$


In [ ]:
def powers1(d: int, t: float):
    return np.pow(t, np.arange(0, d + 1))


def powers(d: int, t: NDArray):
    exps = np.arange(d + 1)
    return t[..., None] ** exps[None, ...]

#### The $\begin{bmatrix} B_{i-d} & \cdots & B_{i} \end{bmatrix}$

$= P^{(d)} M^{(d)}$


In [ ]:
def splines1(d: int, M: NDArray, t: float):
    """[B_i-d ... B_i] = [0, 1, ... t^d] M"""
    return powers1(d, t) @ M


def splines(d: int, M: NDArray, t: NDArray):
    """[B_i-d ... B_i] = [0, 1, ... t^d] M"""
    return powers(d, t) @ M

#### The $C^{(d)}_i$

$= \begin{bmatrix}
c_{i-d} \\
c_{i-d+1} \\
\vdots \\
c_{i}
\end{bmatrix}$


In [ ]:
def cpoints(d: int, points: NDArray, i: int):
    """Slice of control points for i'th segment"""
    assert i >= d and i < len(points)
    return points[i - d : i + 1]

The
$\mathcal{C}^{(d)}_i(t) = P^{(d)} M^{(d)} C^{(d)}_i$


In [ ]:
def segm_curve(d: int, M: NDArray, points: NDArray, i: int, tt: NDArray):
    return powers(d, tt) @ M @ cpoints(d, points, i)

The $\mathcal{C}^{(d)}$

$ = P^{(d)} M^{(d)} \big[ C^{(d)}_i |_d^n \big]$


In [ ]:
def mega_curve(d, M, points, tt):
    pm = powers(d, tt) @ M
    return np.concat([pm @ cpoints(d, points, j) for j in range(d, len(points))])

---


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
def mycurve(points):
    return mega_curve(3, M3, points, np.arange(0, 1.0, 0.125))

In [ ]:
points = arr([
    np.arange(0, 10),
    np.random.random(10) * 5,
    np.random.random(10) * 5,
]).T

In [ ]:
pvcontrol = pv.lines_from_points(points)

In [ ]:
pl = pv.Plotter()
pl.background_color = pv.Color("#303030")
pl.show_grid()
pl.add_mesh(pvcontrol, color="gray", line_width=2)
pl.view_xy()
pl.show()

In [ ]:
def plotcurve(points):
    acurve = mycurve(points)
    lng = acurve.shape[0]
    pvcurve = pv.PolyData(acurve, lines=[lng, *range(lng)])
    pl.add_mesh(pvcurve.tube(radius=0.0625), color=tab10colors[0], smooth_shading=True, name="thecurve")

In [ ]:
plotcurve(points)

In [ ]:
def update(point, idx):
    points[idx] = point
    pvcontrol.points = points
    plotcurve(points)

In [ ]:
pl.add_sphere_widget(update, center=pvcontrol.points, radius=0.125)
None